# 課程 17 - 使用 Foundry Local 和 Qwen 建立本地 AI 代理

在這個筆記本中，你會建立一個<strong>本地工程助理</strong>，完全在你的工作站上運行。整個過程不使用任何雲端推理。該助理可以：

1. <strong>透過 Foundry Local 以 Qwen 函數呼叫</strong>調用工具。
2. <strong>列出並閱讀</strong>沙盒專案目錄中的檔案。
3. <strong>使用簡單指標分析</strong>程式碼。
4. <strong>使用本地 RAG（Chroma）搜尋</strong>文件。
5. **使用本地 MCP 伺服器**（若未設定則會自動跳過）。

代理程式碼與雲端課程幾乎相同 — 唯一不同的是客戶端端點從雲端移至 `localhost`。


## 設定

在執行此筆記本前：

1. **安裝 Microsoft Foundry Local**（查看您的作業系統的[文件](https://learn.microsoft.com/azure/ai-foundry/foundry-local/)）。
2. **下載並啟動 Qwen 模型：**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. 安裝以下 Python 套件。

所有程式在本機運行。需要大約 8 GB RAM 的機器是比較現實的最低要求。


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. 連接到 Foundry Local

`FoundryLocalManager` 如有需要會下載模型，啟動本地服務，並提供我們一個 **與 OpenAI 相容的端點**。然後我們將標準的 OpenAI SDK 指向該端點。API 金鑰是一個本地的佔位符 — 不涉及任何雲端憑證。


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. 本地工具（受限制的檔案操作）

我們在磁碟上建立一個小型範例專案，然後定義該專案根目錄範圍內的工具。即使在本地，沙盒檢查仍然很重要：一個讀取任意路徑的工具是以你的使用者權限運行，能接觸你有權限操作的任何項目。


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. 本地 RAG 與 Chroma

我們將一小組文檔片段嵌入至本地 Chroma 集合中。Chroma 在進程內運行並將向量儲存在磁碟上 — 無需伺服器，無需雲端。`search_docs` 工具會檢索與查詢最相關的片段。


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. 工具呼叫循環

現在我們使用 OpenAI 工具 schema 向模型註冊工具，並運行標準的工具呼叫循環 — 模型請求一個工具，我們在本地執行它，將結果回傳，然後重複進行直到模型產生最終答案。Qwen 可靠的函數呼叫讓這個過程能夠在設備上運作。


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. 本地 MCP（可選）

MCP 是一種傳輸，而不是雲端服務 — MCP 伺服器可以作為本地進程透過 `stdio` 運行。以下的單元展示了如果你配置了本地 MCP 伺服器（例如檔案系統伺服器），你如何連接該伺服器。當 `LOCAL_MCP_COMMAND` 未設置時，它會優雅地跳過，因此筆記本仍可端對端運行。

安全提示：本地 MCP 伺服器以你的使用者權限運行。請將它限制在專案目錄內，並在使用其輸出前先驗證。


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## 摘要

你建構了一個完全在你的機器上運行的工程助手：

- **Foundry Local** 在一個相容於 OpenAI 的端點後面提供了一個 **Qwen** 模型——這樣代理程式碼就與雲端課程相匹配。
- <strong>沙盒工具</strong> 讓代理可以在不離開專案目錄的情況下存取檔案和進行程式碼分析。
- **Chroma** 提供了文件的 **本地 RAG**。
- **Local MCP** 展示了如何在離線狀態下重用 MCP 生態系統。

全程未使用任何雲端推理。

### 挑戰

擴展本地代理以：

1. **與多個 MCP 伺服器協作** — 連接一個檔案系統伺服器和一個 git 伺服器，讓代理在它們之間選擇。
2. <strong>使用本地記憶</strong> — 將短暫的對話歷史保存到磁碟，讓助手記住筆記本重啟前的早期對話。
3. <strong>支援本地多代理協同</strong> — 新增第二個本地代理（例如審查員），並讓兩者合作完成任務。

在下一課中，你將學習如何保護已部署的 AI 代理。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
